# TBP — Sensibilidad por bandas del parámetro $\bar A$

**Opción 1 de la propuesta del grupo.** En vez de fijar $\bar A$ en un único valor, corremos el modelo para $\bar A \in \{15, 20, 25, 30\}$ años y graficamos una **banda de incertidumbre**.

Contexto: hoy $\bar A$ tiene dos escenarios: 14,2 (observado ANSES, **sesgado a la baja**: ANSES cuenta aportes solo desde jul-1994 y a personas con ≥12 meses de aporte) y 30 (requisito legal de referencia: es el mínimo para jubilarse, no un máximo). La verdad está entre ambos, pero no sabemos dónde.

$$TBP_A = \frac{N_t(1-\mu)\,\bar A\,\tau}{J_{t+65}\,E_r\,\rho},\qquad TBP_C = \frac{N_t(1-\mu)\,\bar A\,\tau}{J_{t+65}\,\rho}\ \text{(años)},\qquad TBP_B=\frac{\bar A\,\tau}{E_r\,\rho}\ \text{(cohorte pura)}$$

Constantes: $\tau = 0{,}2177$ (11 % personal + 10,77 % patronal; Anuario ANSES p. 39) y $\rho = 0{,}403$ (tasa de sustitución ANSES, dic-2023, p. 91). Todo lo demás sale del CSV `tbp_argentina_cohortes_1950_2035.csv`; no se inventa ningún dato.

> **Qué NO hace este notebook:** no corrige el sesgo de $\bar A$ ni lo estima. Es una forma honesta de *mostrar* la incertidumbre de nivel. La estimación va por la Opción 2 (calibración con EPH) y la Opción 3 ($\bar A_t$ dinámico).

## 1. Carga de datos

In [ ]:
import os, io, urllib.request
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

CSV_NAME = "tbp_argentina_cohortes_1950_2035.csv"
RAW_URL = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados/"
CANDIDATAS = ["../datos/procesados", "datos/procesados"]

def cargar_csv():
    for carpeta in CANDIDATAS:
        ruta = os.path.join(carpeta, CSV_NAME)
        if os.path.exists(ruta):
            print("Leyendo:", ruta)
            return pd.read_csv(ruta), carpeta
    print("No encontrado localmente; descargando de", RAW_URL)
    with urllib.request.urlopen(RAW_URL + CSV_NAME) as r:
        df = pd.read_csv(io.BytesIO(r.read()))
    return df, None

d, CARPETA = cargar_csv()
# carpeta de salida: la local si existe; si no, la primera candidata que exista (o ./)
OUT = CARPETA or next((c for c in CANDIDATAS if os.path.isdir(c)), ".")
print("Salidas en:", OUT)
print(d.shape, "| cohortes", d.cohorte.min(), "-", d.cohorte.max())
d.head(3)

## 2. Parámetros y cálculo para cada escenario de $\bar A$

In [ ]:
TAU = 0.11 + 0.1077     # Anuario ANSES p.39
RHO = 0.403             # Anuario ANSES p.91 (dic-2023), constante
A_ESCENARIOS = [15, 20, 25, 30]     # parametrizable
CORTE = 2024.5          # DEIS observado (<=2024) vs ONU proyectado (>=2025)

# N_t * S65 / J = D (factor demografico, ya en el CSV); lo reconstruimos desde las columnas base
d["D_calc"] = d.N_t * d.S65 / d.J
assert np.allclose(d.D_calc, d.D)

filas = []
for A in A_ESCENARIOS:
    t = pd.DataFrame({"cohorte": d.cohorte, "anio_65": d.anio_65, "fuente_N": d.fuente_N, "A": A})
    t["TBP_A"] = d.N_t * d.S65 * A * TAU / (d.J * d.E65 * RHO)
    t["TBP_C"] = d.N_t * d.S65 * A * TAU / (d.J * RHO)
    t["TBP_B"] = A * TAU / (d.E65 * RHO)
    for v in ["TBP_A", "TBP_B", "TBP_C"]:
        base = t.loc[t.cohorte == 1950, v].iloc[0]
        t[v + "_idx"] = 100 * t[v] / base          # indice base cohorte 1950 = 100
    filas.append(t)
sens = pd.concat(filas, ignore_index=True)
sens = sens.rename(columns={"A": "A_barra"})
sens.head()

## 3. Test de control (asserts)

1. **Consistencia con el CSV existente:** nuestro cálculo debe reproducir `TBP_*_legal` (Ā = 30) y `TBP_*_obs` (Ā = 14,2) del CSV.
2. **Linealidad en $\bar A$:** $TBP(2\bar A) = 2\cdot TBP(\bar A)$.
3. **Invarianza del índice normalizado:** el índice base 1950 = 100 es idéntico en todos los escenarios.

In [ ]:
def tbp(A, tipo):
    if tipo == "A": return d.N_t * d.S65 * A * TAU / (d.J * d.E65 * RHO)
    if tipo == "C": return d.N_t * d.S65 * A * TAU / (d.J * RHO)
    if tipo == "B": return A * TAU / (d.E65 * RHO)

# 1) reproduce el CSV existente
for tipo in "ABC":
    assert np.allclose(tbp(30.0, tipo), d[f"TBP_{tipo}_legal"])
    assert np.allclose(tbp(14.2, tipo), d[f"TBP_{tipo}_obs"])
print("OK 1) reproduce TBP_*_legal (A=30) y TBP_*_obs (A=14,2) del CSV")

# 2) linealidad
for tipo in "ABC":
    for A in [7.5, 10, 12.5, 15]:
        assert np.allclose(tbp(2 * A, tipo), 2 * tbp(A, tipo))
print("OK 2) linealidad: TBP(2A) = 2*TBP(A) para A, B y C")

# 3) indice normalizado identico entre escenarios
for v in ["TBP_A_idx", "TBP_B_idx", "TBP_C_idx"]:
    piv = sens.pivot(index="cohorte", columns="A_barra", values=v)
    ref = piv.iloc[:, 0]
    for c in piv.columns:
        assert np.allclose(piv[c], ref)
print("OK 3) el indice normalizado (base 1950=100) es identico entre escenarios")

## 4. Hallazgo clave: $\bar A$ cambia el **nivel**, no la **historia**

Como $\bar A$ entra en el numerador como factor multiplicativo y no depende de la cohorte $t$ (en este modelo es una constante), el TBP es **lineal homogéneo** en $\bar A$:

$$TBP_C(t;\bar A) = \bar A\cdot\underbrace{\frac{N_t(1-\mu_t)\,\tau}{J_{t+65}\,\rho}}_{k_C(t)\ \text{(no depende de }\bar A)}$$

Entonces, para dos escenarios cualesquiera $\bar A_1,\bar A_2$ y dos cohortes $t,s$:

$$\frac{TBP_C(t;\bar A_1)}{TBP_C(s;\bar A_1)}=\frac{\bar A_1\,k_C(t)}{\bar A_1\,k_C(s)}=\frac{k_C(t)}{k_C(s)}=\frac{\bar A_2\,k_C(t)}{\bar A_2\,k_C(s)}=\frac{TBP_C(t;\bar A_2)}{TBP_C(s;\bar A_2)}$$

$\bar A$ se cancela. Lo mismo vale para $TBP_A$ ($k_A=k_C/E_r$) y $TBP_B$. Por lo tanto:

* El **índice normalizado** (cohorte 1950 = 100) es **idéntico** para cualquier $\bar A$: $\text{Idx}(t)=100\,k(t)/k(1950)$.
* Las **variaciones porcentuales**, los **cocientes entre cohortes**, los **puntos de inflexión** y las **tendencias relativas** no dependen de $\bar A$.
* Lo que sí cambia con $\bar A$ es el **nivel**: la curva se estira verticalmente en un factor $\bar A_2/\bar A_1$ (entre $\bar A=15$ y $\bar A=30$, un factor 2).

**Consecuencia para la presentación:** la incertidumbre sobre $\bar A$ es incertidumbre de *nivel*, no de *comportamiento*. La historia (caída por natalidad y longevidad, ver abajo) es robusta; el valor absoluto, no.

> **Advertencia sobre la parte proyectada:** a partir de la cohorte 2025 los nacimientos $N_t$ son proyección ONU, no dato DEIS. La ONU proyecta ~507 mil nacimientos en 2024 vs ~413 mil observados por DEIS; el **rebote posterior a 2024 es un artefacto del supuesto ONU**, no un dato. No debe leerse como recuperación.
>
> **Sobre el nivel:** no interpretamos "TBP = 1" como umbral fiscal; el nivel depende de $\bar A$, de $\tau$ y de $\rho$ constantes, y no de un análisis de sostenibilidad.

## 5. Gráficos

In [ ]:
COL = plt.cm.viridis(np.linspace(0.1, 0.85, len(A_ESCENARIOS)))

def banda(ax, var, titulo, ylabel):
    piv = sens.pivot(index="cohorte", columns="A_barra", values=var)
    ax.fill_between(piv.index, piv.min(axis=1), piv.max(axis=1), color="tab:blue", alpha=0.15,
                    label=f"banda min-max (A = {min(A_ESCENARIOS)} a {max(A_ESCENARIOS)})")
    for c, A in zip(COL, A_ESCENARIOS):
        ax.plot(piv.index, piv[A], color=c, lw=1.6, label=f"A = {A}")
    ax.axvline(CORTE, color="gray", ls="--", lw=1)
    ax.text(CORTE + 0.5, ax.get_ylim()[1] * 0.97, "2024/2025: DEIS | ONU proyectado",
            fontsize=7.5, color="gray", va="top")
    ax.set_title(titulo, fontsize=10); ax.set_ylabel(ylabel); ax.set_xlabel("año de nacimiento de la cohorte")
    ax.grid(alpha=.3); ax.legend(fontsize=7.5, loc="upper right", bbox_to_anchor=(1, 0.88))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
banda(ax[0], "TBP_C", "TBP_C (versión recomendada, en años)\nbanda de incertidumbre de NIVEL", "TBP_C (años)")
banda(ax[1], "TBP_A", "TBP_A (fórmula original, con $E_r$)\nbanda de incertidumbre de NIVEL", "TBP_A")

# Panel (b): indice normalizado
for c, A in zip(COL, A_ESCENARIOS):
    s = sens[sens.A_barra == A]
    ax[2].plot(s.cohorte, s.TBP_C_idx, color=c, lw=3.5 - 0.7 * A_ESCENARIOS.index(A), alpha=.8, label=f"A = {A}")
ax[2].axvline(CORTE, color="gray", ls="--", lw=1)
ax[2].axhline(100, color="gray", ls=":", lw=.8)
ax[2].set_title("Índice normalizado de TBP_C (cohorte 1950 = 100)\nlas 4 curvas se superponen: Ā no cambia la forma", fontsize=10)
ax[2].set_xlabel("año de nacimiento de la cohorte"); ax[2].set_ylabel("índice (1950 = 100)")
ax[2].grid(alpha=.3); ax[2].legend(fontsize=7.5)
ax[2].text(CORTE + 0.5, ax[2].get_ylim()[1] * 0.97, "post-2024: supuesto ONU\n(rebote = artefacto)", fontsize=7.5, color="gray", va="top")

fig.suptitle("Sensibilidad al parámetro Ā (años de aporte promedio): cambia el nivel, no la historia", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(f"{OUT}/tbp_sensibilidad_A.png", dpi=130)
plt.show()

## 6. Resultados: rango de $TBP_C$ por escenario en cohortes seleccionadas

In [ ]:
sel = [1950, 1975, 1990, 2000, 2010, 2024]
tabla = sens[sens.cohorte.isin(sel)].pivot(index="cohorte", columns="A_barra", values="TBP_C").round(3)
tabla["max/min"] = (tabla[max(A_ESCENARIOS)] / tabla[min(A_ESCENARIOS)]).round(2)
display(tabla)

idx = sens[(sens.A_barra == A_ESCENARIOS[0]) & sens.cohorte.isin(sel)].set_index("cohorte")["TBP_C_idx"].round(1)
print("Índice TBP_C (1950=100), igual en todos los escenarios:")
print(idx.to_string())

## 7. Exportar

`tbp_sensibilidad_A.csv`: una fila por cohorte × escenario, con TBP_A, TBP_B, TBP_C en nivel y como índice base 1950 = 100.

In [ ]:
sens.to_csv(f"{OUT}/tbp_sensibilidad_A.csv", index=False)
print("Escrito:", f"{OUT}/tbp_sensibilidad_A.csv", sens.shape)
print("Escrito:", f"{OUT}/tbp_sensibilidad_A.png")

## 8. Lectura y límites

* **Robusto (no depende de Ā):** forma de la curva, caída relativa entre cohortes, tendencias. Se comunica con el índice normalizado.
* **Incierto (sí depende de Ā):** el nivel. Se comunica con la banda 15–30 (factor 2 entre extremos).
* **No resuelto:** el sesgo de $\bar A$. Las cuatro cifras 15/20/25/30 son una grilla ilustrativa, no una distribución de probabilidad: la banda no es un intervalo de confianza.
* **Post-2024:** el rebote es un artefacto del supuesto ONU de nacimientos.
* **Próximos pasos:** Opción 2 (calibrar $\bar A$ con EPH) para acotar el nivel; Opción 3 ($\bar A_t$ dinámico por cohorte) que, a diferencia de esta, **sí** puede cambiar la forma de la curva.